# Parte 6 (opcional) – ¿Le gana una regresión logística al BG/NBD?

**Autor:** Herman Andrés Echeverría Rojas · Grupo 1

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

**Referencias principales**

- Fader, P. S., Hardie, B. G. S. y Lee, K. L. (2005). "Counting Your Customers" the Easy Way: An Alternative to the Pareto/NBD Model. *Marketing Science*, 24(2), 275–284.
- Documentación de scikit-learn: `LogisticRegression`, `StratifiedKFold`, `Pipeline` y `brier_score_loss`.

En la Parte 4, Gregorio encontró que $P(Z = 1)$ del BG/NBD no ordena a los clientes mucho mejor que la recencia sola. Si lo que queremos es $P(Z = 1)$, una regresión logística con $x$, $t_x$ y $T$ es la respuesta obvia de cualquier científico de datos. Aquí la ponemos a competir (6.1) y después revisamos si la competencia fue justa (6.2–6.4). Las funciones del BG/NBD son las de `src/bgnbd.py` (`p_compra`, de Gregorio) y los parámetros se leen de `resultados/params_cdnow.json`.

In [1]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from datos import cargar_cdnow, resumen_clientes
from bgnbd import p_compra

trans = cargar_cdnow("../data/CDNOW_sample.txt")
params = json.load(open("../resultados/params_cdnow.json"))
VARIABLES = ["x", "tx", "T"]

# Calibración hasta el 30 sep 1997 y holdout de 39 semanas, como en las Partes 3 y 4
res = resumen_clientes(trans, "1997-09-30", "1998-06-30")
z = (res["x_star"] >= 1).astype(int).values
res["p_compra"] = p_compra(39, res["x"], res["tx"], res["T"], params)

print(f"{len(res)} clientes; tasa base de Z = {z.mean():.3f}")
print("Parámetros del BG/NBD:", {k: round(v, 4) for k, v in params.items()})

2357 clientes; tasa base de Z = 0.290
Parámetros del BG/NBD: {'r': 0.2426, 'alpha': 4.4136, 'a': 0.7929, 'b': 2.4259}


In [2]:
def nueva_logistica():
    # Sin penalización, como pide el enunciado. El escalador va dentro del pipeline para que,
    # en la validación cruzada, la media y la DE se calculen solo con los pliegues de entrenamiento.
    return make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=1000))


def metricas(y, prob):
    return {"AUC": roc_auc_score(y, prob), "Brier": brier_score_loss(y, prob),
            "prob. promedio": np.mean(prob), "tasa real": np.mean(y)}


def tabla_deciles(prob, y):
    # rank(method="first") rompe empates para que cada decil tenga el mismo número de clientes
    decil = pd.qcut(pd.Series(prob).rank(method="first"), 10, labels=range(1, 11))
    g = pd.DataFrame({"prob": np.asarray(prob), "y": np.asarray(y)}).groupby(decil.values, observed=True)
    return pd.DataFrame({"prob. promedio": g["prob"].mean(), "fracción que compró": g["y"].mean()}).rename_axis("decil")


def grafica_calibracion(tablas, titulo, archivo):
    colores = {"Logística": "#4C72B0", "BG/NBD": "#C44E52"}
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], color="gray", ls=":", label="Calibración perfecta")
    for nombre, t in tablas.items():
        ax.plot(t["prob. promedio"], t["fracción que compró"], marker="o",
                color=colores.get(nombre.split(" ")[0], None), label=nombre)
    ax.set(title=titulo, xlabel="Probabilidad promedio del decil", ylabel="Fracción que compró",
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"../figs/{archivo}", dpi=150)
    plt.show()